# National AI Strategy — NLLB Translation

Translate CJK-heavy national AI strategy text into English with `facebook/nllb-200-distilled-600M`.

The notebook writes translated/cached text files to:

`data/pdf/AI Policy/_translated_nllb/`

English-heavy documents are skipped because they do not need translation. Existing translated files are reused unless `FORCE_RETRANSLATE = True`.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / 'data').is_dir() and (p / 'src').is_dir()
)
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import re

import pandas as pd
import pdfplumber

from ai_policy.common import find_repo_root

repo_root = find_repo_root(Path.cwd())
pdf_dir = repo_root / 'data' / 'pdf' / 'AI Policy'
translated_dir = pdf_dir / '_translated_nllb'
translated_dir.mkdir(parents=True, exist_ok=True)

print(f'Repo root      : {repo_root}')
print(f'PDF directory  : {pdf_dir}')
print(f'Output directory: {translated_dir}')


## 2. Document Mapping

Multiple PDF files can map to the same country. China currently combines the original national strategy PDF and the newer AI+ Action policy PDF.

In [ ]:
from ai_policy.common import STRATEGY_COUNTRY_MAP as COUNTRY_MAP

for pdf_name, country in COUNTRY_MAP.items():
    status = 'ok' if (pdf_dir / pdf_name).exists() else 'missing'
    print(f'{status:7s} {country:15s} {pdf_name}')


## 3. Extract and Aggregate PDF Text

In [ ]:
from ai_policy.text_utils import extract_pdf_text

texts = {}
page_counts = {}
source_files = {}

for pdf_name, country in COUNTRY_MAP.items():
    pdf_path = pdf_dir / pdf_name
    if not pdf_path.exists():
        print(f'Warning: missing PDF for {country}: {pdf_name}')
        continue

    text = extract_pdf_text(pdf_path)
    with pdfplumber.open(pdf_path) as pdf:
        pages = len(pdf.pages)

    if country in texts:
        texts[country] += f'\n\n--- Source: {pdf_path.name} ---\n\n{text}'
        page_counts[country] += pages
        source_files[country].append(pdf_path.name)
    else:
        texts[country] = text
        page_counts[country] = pages
        source_files[country] = [pdf_path.name]

    print(f'{country:15s} {pages:>4d} pages, {len(text):>8,} chars  ({pdf_path.name})')

print('\nAggregated documents:')
for country, text in texts.items():
    print(f'{country:15s} {page_counts[country]:>4d} pages, {len(text):>8,} chars, {len(source_files[country])} source file(s)')



## 4. Translation Helpers

In [ ]:
from ai_policy.text_utils import cjk_ratio, load_nllb, split_translation_units, translate_units_nllb

NLLB_MODEL_NAME = 'facebook/nllb-200-distilled-600M'
NLLB_SOURCE_LANG = 'zho_Hans'
NLLB_TARGET_LANG = 'eng_Latn'

# Reuse existing translated files by default.
FORCE_RETRANSLATE = False

# Paragraphs with CJK ratio below this threshold are copied unchanged.
CJK_RATIO_THRESHOLD = 0.20

# Smaller chunks reduce memory pressure but can lose context.
MAX_CHARS_PER_UNIT = 700
MAX_NEW_TOKENS = 512


## 5. Run NLLB Translation

In [ ]:
translation_stats = []
nllb_tokenizer = None
nllb_model = None
nllb_device = None

for country, text in texts.items():
    units = split_translation_units(text, MAX_CHARS_PER_UNIT)
    cjk_units = sum(cjk_ratio(unit) >= CJK_RATIO_THRESHOLD for unit in units)
    out_path = translated_dir / f'{country}.txt'

    if cjk_units == 0:
        status = 'skipped_no_cjk'
    elif out_path.exists() and not FORCE_RETRANSLATE:
        status = 'cached'
    else:
        if nllb_tokenizer is None:
            print(f'Loading {NLLB_MODEL_NAME}...')
            nllb_tokenizer, nllb_model, nllb_device = load_nllb(NLLB_MODEL_NAME, NLLB_SOURCE_LANG)
            print(f'NLLB device: {nllb_device}')

        print(f'Translating {country}: {cjk_units}/{len(units)} CJK-heavy units')
        translated_text = '\n\n'.join(
            translate_units_nllb(
                units,
                nllb_tokenizer,
                nllb_model,
                nllb_device,
                NLLB_TARGET_LANG,
                CJK_RATIO_THRESHOLD,
                MAX_NEW_TOKENS,
            )
        )
        out_path.write_text(translated_text, encoding='utf-8')
        status = 'translated'

    translation_stats.append({
        'Country': country,
        'Source files': '; '.join(source_files.get(country, [])),
        'Translation file': out_path.name,
        'Units': len(units),
        'CJK-heavy units': cjk_units,
        'Status': status,
    })

translation_stats_df = pd.DataFrame(translation_stats).set_index('Country')
translation_stats_df.to_csv(translated_dir / 'translation_stats.csv')
display(translation_stats_df)


## 6. Inspect Output

In [ ]:
print(f'Saved translations to: {translated_dir}')
for path in sorted(translated_dir.glob('*')):
    print(f'  - {path.name}')

china_path = translated_dir / 'China.txt'
if china_path.exists():
    print('\nChina translation preview:')
    print(china_path.read_text(encoding='utf-8')[:2000])
